# Task 3: Publish CPG Events sang Apache Kafka

Tài liệu này ghi nhận quá trình cấu hình, khởi tạo các topic và kiểm nghiệm tích hợp hệ thống publish event streaming của CPG Parser Service lên Apache Kafka broker chạy local.

## Kiến Trúc Kafka Event Streaming
- **Broker**: Apache Kafka chạy single-node ở chế độ KRaft (không ZooKeeper) trên Docker.
- **Topics**:
  - `cpg.nodes`: Chứa `NODE_UPSERT` và `NODE_DELETE`.
  - `cpg.edges`: Chứa `EDGE_UPSERT` và `EDGE_DELETE`.
  - `source.metadata`: Chứa `FILE_METADATA_UPSERT`.
  - `parser.errors`: Chứa `PARSER_ERROR` (Topic chứa các sự kiện lỗi nghiệp vụ).
- **Partition Key**: Sử dụng `file_id` làm khóa phân vùng để đảm bảo tính nhất quán phân vùng theo từng topic (per-topic partition consistency).
- **Guarantees**:
  - `acks=all` và producer có cấu hình `enable.idempotence=True`. Lưu ý rằng `acks=all` yêu cầu acknowledgement từ toàn bộ in-sync replicas. Trong môi trường Task 3 chỉ có một broker và replication factor bằng 1, nên acknowledgement chỉ đến từ broker duy nhất và không tạo broker redundancy hoặc high availability.
  - SQLite state database chỉ được commit sau khi nhận được delivery acknowledgement từ Kafka.


## 1. Thiết Lập Môi Trường & Import Thư Viện

In [1]:
import os
import subprocess
import json
import shutil
from pathlib import Path

# Resolve PROJECT_ROOT using git
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

print("PROJECT_ROOT:", PROJECT_ROOT)

def get_current_offsets(bootstrap_servers="localhost:9092"):
    from confluent_kafka import Consumer, TopicPartition
    import yaml
    with open(PROJECT_ROOT / "config/topics.yaml", "r") as f:
        config_data = yaml.safe_load(f)
    topics = [t["name"] for t in config_data["topics"] if t["name"] != "connector.errors"]
    partitions_count = {
        "cpg.nodes": 3,
        "cpg.edges": 3,
        "source.metadata": 1,
        "parser.errors": 1
    }
    conf = {
        "bootstrap.servers": bootstrap_servers,
        "group.id": "offset-capturer-temp",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": "false",
    }
    consumer = Consumer(conf)
    offsets = {}
    for topic in topics:
        offsets[topic] = {}
        p_count = partitions_count.get(topic, 1)
        for p in range(p_count):
            tp = TopicPartition(topic, p)
            try:
                low, high = consumer.get_watermark_offsets(tp, timeout=5.0)
                offsets[topic][p] = high
            except Exception:
                offsets[topic][p] = 0
    consumer.close()
    return offsets


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming


## 2. Kiểm Trạng Thái Kafka Broker

In [2]:
# Verify that Kafka container is running and healthy
res_ps = subprocess.run(
    ["docker", "compose", "-f", str(PROJECT_ROOT / "infra/docker-compose.yml"), "ps"],
    check=True,
    capture_output=True,
    text=True
)
print(res_ps.stdout)


NAME        IMAGE                         COMMAND                  SERVICE   CREATED        STATUS                 PORTS
cpg-kafka   confluentinc/cp-kafka:7.4.0   "/etc/confluent/dock…"   kafka     19 hours ago   Up 5 hours (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp



## 3. Khởi Tạo Topics Idempotent

In [3]:
# Run topic creation script
res_topics = subprocess.run(
    [str(PROJECT_ROOT / "scripts/create_topics.sh")],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)
print(res_topics.stdout)


Waiting for Kafka broker to be healthy...
Kafka broker is healthy.
Creating and validating topics...
[OK] Topic 'cpg.nodes' matches desired configuration.
[OK] Topic 'cpg.edges' matches desired configuration.
[OK] Topic 'source.metadata' matches desired configuration.
[OK] Topic 'parser.errors' matches desired configuration.
[OK] Topic 'connector.errors' matches desired configuration.

=== Existing Topics ===
__consumer_offsets
connector.errors
cpg.edges
cpg.nodes
parser.errors
source.metadata


[SUCCESS] All topics match desired configurations.



## 4. Reset Smoke State cho Kafka Run
Chúng ta sẽ sử dụng một SQLite state database biệt lập đặt dưới thư mục verification tạm `workspace/tmp/task3-kafka-verification/` để theo dõi trạng thái chạy của từng Phase.

In [4]:
VERIFICATION_ROOT = PROJECT_ROOT / "workspace/tmp/task3-kafka-verification"
KAFKA_SMOKE_STATE = VERIFICATION_ROOT / "state/graph_smoke.sqlite3"

# Preflight cleanup of isolated directory
if VERIFICATION_ROOT.exists():
    shutil.rmtree(VERIFICATION_ROOT)
VERIFICATION_ROOT.mkdir(parents=True, exist_ok=True)
(VERIFICATION_ROOT / "state").mkdir(exist_ok=True)
(VERIFICATION_ROOT / "offsets").mkdir(exist_ok=True)

print("Isolated verification directory initialized.")

Isolated verification directory initialized.


## 5. Phase A: Fresh Publish Run
Chạy Parser Service ở live mode (`--no-dry-run`) phân tích 1 tệp tin mẫu và publish các sự kiện mới lên Kafka. Chúng ta sử dụng `.github/scripts/assign_reviewers.py` làm tệp mẫu.

In [5]:
# Phase A: Fresh publish run of a semantic smoke file
smoke_file = ".github/scripts/assign_reviewers.py"

# 1. Capture start offsets
start_offsets_a = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_a_start.json", "w") as f:
    json.dump(start_offsets_a, f)

# 2. Run parse-file
env = {**os.environ, "PARSER_STATE_DB": str(KAFKA_SMOKE_STATE)}
res_a = subprocess.run(
    [
        "uv", "run", "lab04", "parse-file",
        "--file", smoke_file,
        "--no-dry-run"
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print(res_a.stdout)

# 3. Capture end offsets
end_offsets_a = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_a_end.json", "w") as f:
    json.dump(end_offsets_a, f)

Parsing single file: .github/scripts/assign_reviewers.py
File processed. Status: SUCCESS, content_hash: 062c8d29b9808501e7fa59d2ad9de8113d9e21c340bf87b40fee71d10dfd3647



## 6. Phase A: Inspect & Validate Messages từ Kafka
Tiến hành consume các message được tạo ra trong Phase A, kiểm tra partition key (`file_id`) và validate cấu trúc bằng JSON Schema.

In [6]:
# Query expected file_id from state db
import sqlite3
from pathlib import Path
conn = sqlite3.connect(KAFKA_SMOKE_STATE)
cursor = conn.cursor()
cursor.execute("SELECT file_id FROM file_state LIMIT 1")
row = cursor.fetchone()
conn.close()
expected_file_id = row[0] if row else "unknown"

# Run inspector for Phase A
res_inspect_a = subprocess.run(
    [
        "uv", "run", "python", "scripts/inspect_kafka_events.py",
        "--start-offsets", str(VERIFICATION_ROOT / "offsets/phase_a_start.json"),
        "--end-offsets", str(VERIFICATION_ROOT / "offsets/phase_a_end.json"),
        "--expected-file-id", expected_file_id
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect_a.stdout)

Assigning specific partitions and seeking to offsets: [TopicPartition{topic=cpg.nodes,partition=0,offset=1188,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=1,offset=50,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=2,offset=22799,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=0,offset=1490,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=1,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=2,offset=28772,leader_epoch=None,error=None}, TopicPartition{topic=source.metadata,partition=0,offset=24,leader_epoch=None,error=None}, TopicPartition{topic=parser.errors,partition=0,offset=19,leader_epoch=None,error=None}]
Listening for messages... (will auto-stop after 5s of inactivity)
[cpg.edges] Part:0 Off:1490 Key:9a58fe92e64589dcb571f84ec339bffd4a0898c5b66c72a047521a76e8426dfe Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:0 Off:1491 Key:9a58fe92

## 7. Phase B: Unchanged Rerun
Chạy lại phân tích tệp tin cũ `.github/scripts/assign_reviewers.py` sử dụng cùng isolated SQLite state database của Phase A. Hệ thống phải nhận biết tệp tin không đổi (`SKIPPED_UNCHANGED`) và không gửi bất kỳ sự kiện đồ thị nào lên Kafka.

In [7]:
# Phase B: Unchanged rerun of the same file
# 1. Capture start offsets
start_offsets_b = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_b_start.json", "w") as f:
    json.dump(start_offsets_b, f)

# 2. Run parse-file
res_b = subprocess.run(
    [
        "uv", "run", "lab04", "parse-file",
        "--file", smoke_file,
        "--no-dry-run"
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print(res_b.stdout)

# 3. Capture end offsets
end_offsets_b = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_b_end.json", "w") as f:
    json.dump(end_offsets_b, f)

Parsing single file: .github/scripts/assign_reviewers.py
File processed. Status: SKIPPED_UNCHANGED, content_hash: 062c8d29b9808501e7fa59d2ad9de8113d9e21c340bf87b40fee71d10dfd3647



## 8. Phase B: Inspect & Verify Zero Graph Update
Tiến hành consume từ Kafka trong khoảng offset của Phase B. Kết quả mong đợi là không có bất kỳ tin nhắn mới nào được gửi lên.

In [8]:
# Run inspector for Phase B: we expect 0 new messages
res_inspect_b = subprocess.run(
    [
        "uv", "run", "python", "scripts/inspect_kafka_events.py",
        "--start-offsets", str(VERIFICATION_ROOT / "offsets/phase_b_start.json"),
        "--end-offsets", str(VERIFICATION_ROOT / "offsets/phase_b_end.json")
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect_b.stdout)

Assigning specific partitions and seeking to offsets: [TopicPartition{topic=cpg.nodes,partition=0,offset=1782,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=1,offset=50,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=2,offset=22799,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=0,offset=2235,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=1,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=2,offset=28772,leader_epoch=None,error=None}, TopicPartition{topic=source.metadata,partition=0,offset=25,leader_epoch=None,error=None}, TopicPartition{topic=parser.errors,partition=0,offset=19,leader_epoch=None,error=None}]
Listening for messages... (will auto-stop after 5s of inactivity)

=== Verification Window ===
topic=cpg.edges partition=0 start=2235 end=2235
topic=cpg.edges partition=1 start=0 end=0
topic=cpg.edges partition=2 start=28772 end=28772
topic=cpg.nodes par

## 9. Phase C: Kiểm Nghiệm Flow Lỗi Cú Pháp (Parser Error Event)
Chúng ta sẽ phân tích một tệp tin Python chứa lỗi cú pháp để kiểm tra xem `PARSER_ERROR` event có được đẩy vào topic `parser.errors` hay không, và đảm bảo SQLite state store không commit bản ghi thành công cho tệp tin này.

In [9]:
# Copy broken python syntax file to target repo using path semantic: _verification/broken_syntax.py
target_dir = PROJECT_ROOT / "workspace/source/transformers-pr-agent/_verification"
target_path = target_dir / "broken_syntax.py"

# Preflight cleanup
if target_path.exists():
    with open(target_path, "rb") as f:
        content = f.read()
    if b"class BrokenSyntax" in content or len(content) < 500:
        target_path.unlink()

# Capture start offsets
start_offsets_c = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_c_start.json", "w") as f:
    json.dump(start_offsets_c, f)

# Ensure target_dir exists
target_dir.mkdir(parents=True, exist_ok=True)

try:
    if target_path.exists():
        raise RuntimeError(f"Fixture path {target_path} already exists and cannot be overwritten")

    shutil.copy(PROJECT_ROOT / "tests/fixtures/broken_syntax.py", target_path)

    # Run parser on broken syntax file
    res_parse_err = subprocess.run(
        [
            "uv", "run", "lab04", "parse-file",
            "--file", "_verification/broken_syntax.py",
            "--no-dry-run"
        ],
        capture_output=True,
        text=True,
        cwd=str(PROJECT_ROOT),
        env=env
    )
    print(res_parse_err.stdout)

finally:
    # Cleanup fixture file and dir
    if target_path.exists():
        target_path.unlink()
    if target_dir.exists() and not any(target_dir.iterdir()):
        target_dir.rmdir()

# Capture end offsets
end_offsets_c = get_current_offsets()
with open(VERIFICATION_ROOT / "offsets/phase_c_end.json", "w") as f:
    json.dump(end_offsets_c, f)

Parsing single file: _verification/broken_syntax.py
File processed. Status: FAILED, content_hash: c66248af053766c7e02b9681da497cc4e2e6041914782231a74d41e60929eea9
Error details: SyntaxError in _verification/broken_syntax.py: invalid syntax at line 3, col 10



## 10. Consume & Validate Parser Error Event
Consume từ `parser.errors` topic để lấy `PARSER_ERROR` event và validate cấu trúc.

In [10]:
# Compute expected error file_id dynamically
import sys
from pathlib import Path
sys.path.append(str(PROJECT_ROOT / "src"))
from parsing.identifiers import IdentifierGenerator
expected_error_file_id = IdentifierGenerator.generate_file_id("huggingface/transformers-pr-agent", Path("_verification/broken_syntax.py"))

res_inspect_c = subprocess.run(
    [
        "uv", "run", "python", "scripts/inspect_kafka_events.py",
        "--start-offsets", str(VERIFICATION_ROOT / "offsets/phase_c_start.json"),
        "--end-offsets", str(VERIFICATION_ROOT / "offsets/phase_c_end.json"),
        "--expected-error-file-id", expected_error_file_id
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect_c.stdout)

Assigning specific partitions and seeking to offsets: [TopicPartition{topic=cpg.nodes,partition=0,offset=1782,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=1,offset=50,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=2,offset=22799,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=0,offset=2235,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=1,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=2,offset=28772,leader_epoch=None,error=None}, TopicPartition{topic=source.metadata,partition=0,offset=25,leader_epoch=None,error=None}, TopicPartition{topic=parser.errors,partition=0,offset=19,leader_epoch=None,error=None}]
Listening for messages... (will auto-stop after 5s of inactivity)
[parser.errors] Part:0 Off:19 Key:c0a6cbf4dacc775e7a368d5cad3c3a5d6e062f2680136425db69ae3f32c90f46 Event:PARSER_ERROR
  [OK] Schema validation passed.

=== Verification Window ===
topic=cp

## 11. Xác Minh Transaction Boundary
Xác minh xem state database của `broken_syntax.py` đã bị commit hay chưa (kết quả mong đợi là không tồn tại bản ghi trong database).

In [11]:
# Load state from state store using sqlite
import sqlite3
conn = sqlite3.connect(KAFKA_SMOKE_STATE)
cursor = conn.cursor()
cursor.execute("SELECT * FROM file_state WHERE file_path LIKE '%_verification/broken_syntax.py%'")
rows = cursor.fetchall()
conn.close()

print("Database committed rows for broken_syntax.py:", len(rows))
assert len(rows) == 0, "Error: State must not be committed for syntax error file"
print("SUCCESS: Transaction boundary verified.")

Database committed rows for broken_syntax.py: 0
SUCCESS: Transaction boundary verified.


## 12. Dọn Dẹp Tài Nguyên Tạm Thời
Dọn dẹp các tệp tin chứa offset JSON và thư mục verification tạm thời để không ảnh hưởng đến các lần chạy sau và giữ cho git tree luôn sạch.

In [12]:
# Clean up temporary offset files
shutil.rmtree(VERIFICATION_ROOT, ignore_errors=True)

# Verify clean git status of target repo
git_status = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT / "workspace/source/transformers-pr-agent"), "status", "--short"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

if git_status:
    print(f"Warning: Source repository is dirty: {git_status}")
    raise RuntimeError("Source repository is dirty after verification run")
else:
    print("Source repository is clean. Resource cleanup successfully completed.")

Source repository is clean. Resource cleanup successfully completed.


## Ordering Guarantees & Downstream Ingestion Strategy

### 1. Cơ chế đảm bảo thứ tự hiện có (Actual Guarantees)
- **Khóa phân vùng (`file_id`)**: Đảm bảo tất cả các event có cùng `file_id` trong cùng một topic sẽ luôn đi vào **cùng một partition**. Điều này giúp giữ nguyên thứ tự xuất bản (offset order) cho các event của cùng một file trên topic đó.
- **Không có thứ tự chéo topic (No Cross-Topic Ordering)**: Vì các event được định tuyến sang các topic khác nhau (`cpg.nodes`, `cpg.edges`, `source.metadata`), Kafka **không đảm bảo** bất kỳ thứ tự phân phối nào giữa các topic này. Consumer (hoặc Kafka Connect Sink) có thể nhận tin nhắn từ `cpg.edges` trước khi nhận tin nhắn tương ứng từ `cpg.nodes`.
- **Vai trò của Metadata**: Tin nhắn `FILE_METADATA_UPSERT` được gửi sau cùng trong code của Parser Service nhưng **không** tạo thành một completion barrier ở downstream vì các tin nhắn ở các topic khác có thể đến sau hoặc được xử lý không đồng bộ.

### Ordering guarantees của kiến trúc hiện tại

| Phạm vi | Kafka có bảo đảm không? | Giải thích |
|---|---|---|
| Thứ tự event của cùng file trong một partition của `cpg.nodes` | **Có** | `file_id` giúp các node event của cùng file được định tuyến nhất quán vào một partition; Kafka bảo toàn thứ tự offset trong partition đó. |
| Thứ tự event của cùng file trong một partition của `cpg.edges` | **Có** | `file_id` giúp các edge event của cùng file được định tuyến nhất quán vào một partition; Kafka bảo toàn thứ tự offset trong partition đó. |
| Thứ tự giữa `cpg.nodes` và `cpg.edges` | **Không được Kafka bảo đảm** | Hai topic có partition và offset độc lập. Edge có thể được downstream xử lý trước node. |
| Thứ tự giữa graph topics và `source.metadata` | **Không được Kafka bảo đảm** | Application publish metadata sau graph events nhưng downstream completion order giữa các topic vẫn độc lập. |
| Một thứ tự toàn cục cho toàn bộ event của một file | **Không tồn tại trong thiết kế hiện tại** | Event của cùng file được chia ra nhiều topic nên không có một offset hoặc transaction order chung. |

*Lưu ý: Các dòng "Không được Kafka bảo đảm" không phản ánh kiểm thử thất bại, mà mô tả giới hạn vật lý của kiến trúc multi-topic. Tầng downstream (Neo4j ingestion ở Task 4) phải được thiết kế và kiểm chứng để không phụ thuộc vào thứ tự đến giữa node và edge events (chịu được edge-before-node và các delete races).*

---

## Phân biệt Error Topics và DLQ (Error Topic Semantics)

Trong hệ thống, chúng ta phân biệt rõ rệt hai kênh xử lý lỗi (failure channels) độc lập để phục vụ cho các mục đích giám sát khác nhau:

| Topic | Phân loại (Classification) | Producer (Nguồn tạo) | Mục đích (Purpose) |
|---|---|---|---|
| `parser.errors` | Parser business error topic | Parser Service | Chứa `PARSER_ERROR` khi phân tích source file thất bại (ví dụ: lỗi cú pháp Python). |
| `connector.errors` | Kafka Connect Dead Letter Queue (DLQ) | Kafka Connect | `connector.errors` là planned Kafka Connect dead-letter topic dành cho các source records mà Neo4j Sink Connector không xử lý thành công. |

### 1. parser.errors (Business Error Topic)
- Đây **không phải** là Dead Letter Queue (DLQ).
- Sự kiện `PARSER_ERROR` được tạo chủ động bởi mã nguồn của Parser Service dựa trên nghiệp vụ phân tích tĩnh, có schema sự kiện xác thực rõ ràng.
- Các lỗi cấu trúc (schema validation failure) trên payload gốc sẽ không được đưa vào topic này để tránh gây nhiễu và đứt gãy luồng xử lý.

### 2. connector.errors (Kafka Connect DLQ)
- Kênh dead-letter phục vụ cho tầng Kafka Connect (sẽ được cấu hình và kiểm chứng ở Task 4).
- Tự động bắt các bản ghi bị từ chối hoặc gặp lỗi ghi bởi Neo4j Sink Connector để giữ an toàn dữ liệu và phục vụ audit.

---

## TASK 3 VERDICT: IMPLEMENTED AND VERIFIED LOCALLY
Task 3 correctness and verification hardening completed in the current local single-broker environment.

### Bằng chứng xác minh (Evidence Checklist)
| Bằng chứng | Trạng thái | Ghi chú |
|---|---|---|
| Cấu hình các topic hiện có (Configured topics exist) | **Passed** | Đã cấu hình và khởi tạo các topic: `cpg.nodes`, `cpg.edges`, `source.metadata`, `parser.errors` và topic dự phòng `connector.errors`. |
| Xác thực schema và định tuyến key phân vùng | **Passed** | Mọi sự kiện đều vượt qua Schema Validation và định tuyến nhất quán theo `file_id`. |
| Xác thực parser error topic | **Passed** | Khi gặp lỗi cú pháp, sự kiện `PARSER_ERROR` được chuyển hướng về `parser.errors` và state database không bị commit. |
| Kafka Connect DLQ runtime verification | **Chưa thực hiện** | `connector.errors` là topic DLQ dự kiến cho Kafka Connect ở Task 4; hành vi cấu hình và kiểm chứng thực tế DLQ chưa được thực hiện trong Task 3. |

---

## Reflection

- **Độ tin cậy**: Việc triển khai biên an toàn "Publish-before-Commit" đảm bảo SQLite local state store luôn đồng bộ chính xác với những gì Kafka broker đã thực sự ghi nhận.
- **Biên giới an toàn và Tính nhất quán**: Crash sau Kafka acknowledgement nhưng trước SQLite commit có thể khiến cùng một batch được publish lại. Stable deterministic IDs tạo cơ sở để Task 4 triển khai idempotent database writes; duplicate handling chưa được kiểm chứng trong Task 3.
- **Tính thực tế trong Downstream Ingestion**: Task 3 không cung cấp consistency toàn hệ thống. Order-tolerant ingestion, idempotent mutations và stale-event protection phải được triển khai và kiểm chứng ở Task 4.
